In [ ]:
# 1. 初始化大模型客户端 (用国内的 DeepSeek)
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key= os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)


In [ ]:
def calculator(expression):
    return eval(expression)

In [ ]:
def get_number():
    return 100

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "需要计算的数学表达式，例如 123 * 456"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
            "type": "function",
            "function": {
                "name": "get_number",
                "description": "得到数字100",
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            }
        }
]

In [ ]:
tool_registry={
    "calculator":calculator,
    "get_number":get_number
    }


In [ ]:
class Agent:
    def __init__(self,client,tools,tool_registry):
        self.client=client
        self.tools=tools
        self.tool_registry=tool_registry
        #Agent自己的对话状态
        self.messages=[]

    
    def run(self,user_input):
        #1，保存用户输入
        self.messages.append(
            {
                "role":"user",
                "content":user_input
            }
        )
        #2，重复调用llm，知道确认不再调用工具后返回最终结果。
        while True:
            print("===== 新一轮 LLM =====")
            response = self.client.chat.completions.create(
                model="deepseek-chat",
                messages=self.messages,
                tools=self.tools
            )    

            message = response.choices[0].message

            #确认没有调用工具，记录并退出循环
            if not message.tool_calls:
                self.messages.append(
                    {
                        "role":"assistant",
                        "content": message.content
                    }   
                )

                return message.content

            #调用工具，增加调用工具的记录。
            self.messages.append(
                {
                    "role":"assistant",
                    "tool_calls":message.tool_calls
                }
            )

            #执行工具生成结果并记录
            for tool_call in message.tool_calls:
                tool_name=tool_call.function.name
                print("本轮工具调用：", tool_name)
                arguments = json.loads(tool_call.function.arguments)

                tool=self.tool_registry[tool_name]
                result=tool(**arguments)

                self.messages.append(
                    {
                        "role":"tool",
                        "tool_call_id": tool_call.id,
                        "content":str(result)
                    }
                )



In [ ]:
agent = Agent(
    client=client,
    tools=tools,
    tool_registry=tool_registry
)

In [ ]:
print(agent.run("获取一个数字，然后给它乘以5"))

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

In [ ]:
'''embedding：参考文档和问题向量化，并匹配最优回复'''
import math
documents = [
    "Python 是一种编程语言",
    "北京是中国的首都",
    "机器学习是人工智能的重要分支",
    "Python 可以用于开发 AI Agent"
]

embeddings=model.encode(documents)
print(embeddings.shape)

query="python能做什么"
embedding_q=model.encode(query)

def cosine_similarity(a, b):
    '''余弦相似度函数'''
    dot_product = sum(x * y for x, y in zip(a, b))

    norm_a = math.sqrt(sum(x ** 2 for x in a))
    norm_b = math.sqrt(sum(x ** 2 for x in b))

    return dot_product / (norm_a * norm_b)

results = []
for i,embedding in enumerate(embeddings):
    score=cosine_similarity(embedding_q, embedding)
    tuple_i=(score,documents[i])
    results.append(tuple_i)

results.sort(key=lambda x: x[0], reverse=True)       
top_3 = results[:3]

#提取文档并上传给llm
contexts=list(map(lambda x:x[1],top_3))
context="\n".join(contexts)
print(context)


In [ ]:
'''最优回复和向量问题丢给llm生成最终回答'''
prompt=f'''
请根据参考资料回答问题

参考资料：
{context}

问题：
{query}
'''

response=client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {
                        "role":"user",
                        "content":prompt
                    }
                ]
            )    
print(response.choices[0].message.content)

In [ ]:
import math
'''chunking:文档切块'''
text = """
Python 是一种高级编程语言。
Python 具有简单易学的特点。
Python 可以用于 Web 开发。
Python 可以用于数据分析。
Python 也可以用于人工智能开发。
Python 可以用于开发 AI Agent。
"""

chunks = text.split("。")
chunks = [chunk.strip() for chunk in chunks if chunk.strip()]
print(chunks)

embeddings=model.encode(chunks)
print(embeddings.shape)
query = "Python 能做什么？"
embedding_query=model.encode(query)

def cosine_similarity(a, b):
    '''余弦相似度函数'''
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x ** 2 for x in a))
    norm_b = math.sqrt(sum(x ** 2 for x in b))

    return dot_product / (norm_a * norm_b)

results=[]
for i,embedding in enumerate(embeddings):
    score=cosine_similarity(embedding_query, embedding)
    tuple_i=(score,chunks[i])
    results.append(tuple_i)

results.sort(key=lambda x:x[0],reverse=True)
top_3=results[:3]



    




In [ ]:
"""将最优回复丢给llm生成最终回复"""
contexts=list(map(lambda x:x[1],top_3))
context="\n".join(contexts)
print(context)

prompt=f'''
请根据参考资料回答问题

问题：
{query}

参考资料：
{context}
'''


response=client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
text = """
Python 是一种高级编程语言。
Python 具有简单易学的特点。
Python 可以用于 Web 开发。
Python 可以用于数据分析。
Python 也可以用于人工智能开发。
Python 可以用于开发 AI Agent。
Agent 可以通过调用工具完成任务。
Agent 也可以使用 RAG 检索知识库中的资料。
"""

chunk_size=30
overlap=10

chunks=[]

for i in range(0,len(text),chunk_size-overlap):
    chunk=text[i:i+chunk_size]
    chunks.append(chunk)
print(chunks)


In [ ]:
sentences = text.split("。")
sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

chunk_size = 30
chunks = []
current_chunk = []

for sentence in sentences:

    sentence = sentence + "。"

    # 当前 chunk 加入这句话后，如果没有超过长度限制
    if len("".join(current_chunk)) + len(sentence) <= chunk_size:
        current_chunk.append(sentence)

    else:
        # 1. 先保存当前 chunk
        chunks.append(current_chunk)

        # 2. 保留当前 chunk 最后一个句子，作为 overlap
        chunks_last = current_chunk[-1:]

        # 3. 新 chunk 从 overlap 开始，再加入当前这句话
        current_chunk = chunks_last + [sentence]

# 循环结束后，保存最后一个 chunk
if current_chunk:
    chunks.append(current_chunk)

print(chunks)




In [ ]:
import math
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

In [ ]:
'''chunks和query的向量化,并取相似度最高的3个值'''

def cosine_similarity(a, b):
    '''余弦相似度函数'''
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x ** 2 for x in a))
    norm_b = math.sqrt(sum(x ** 2 for x in b))

    return dot_product / (norm_a * norm_b)

chunks = ["".join(chunk) for chunk in chunks]
query="Python 能做什么？"

embeddings = model.encode(chunks)
embedding_q=model.encode(query)

print(embeddings.shape)

#相似度计算
results=[]
for i, embedding in enumerate(embeddings):
    score=cosine_similarity(embedding_q,embedding)
    tuple_i=(score,chunks[i])
    results.append(tuple_i)
results.sort(key=lambda x:x[0],reverse=True)
top_3=results[:3]
print(top_3)


'''top_k传给llm生成回答'''

context = "\n".join(x[1] for x in top_3)
print(context)
prompt=f'''
请根据参考文件回答问题

参考文件：
{context}

问题：
{query}
'''

#调用llm
response=client.chat.completions.create(
    model="deepseek-chat",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
        
)

print(response.choices[0].message.content)


In [ ]:
def split_text(text, chunk_size, overlap):
    '''文档切块函数'''

    #1,检查传入参数是否符合规范
    if not isinstance(text, str):
        raise TypeError("text 必须是字符串")

    if not isinstance(chunk_size,int) or isinstance(chunk_size,bool):
        raise TypeError("chunk_size 必须是整数")

    if not isinstance(overlap,int) or isinstance(overlap,bool):
            raise TypeError("overlap 必须是整数")
    
    if chunk_size <= 0:
        raise ValueError("chunk_size 必须大于 0")

    if not 0<=overlap<chunk_size:
        raise ValueError("overlap 必须满足 0 <= overlap < chunk_size")
    
    #2,文档切块（chunking）
    chunks=[]
    step=chunk_size-overlap

    for i in range(0,len(text),step):
        chunk=text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks

'''测试代码'''
try:
    print(split_text("ABCDEFGHIJKLMNO", 5, 2))

except TypeError as e:
     print(f"参数类型错误：{e}")

except ValueError as e:
     print(f"参数值错误：{e}")


def split_by_sentence(text, chunk_size):

    # 1. 按句号切分
    sentences=text.split("。")

    # 2. 去掉空字符串
    sentences=[sentence.strip()for sentence in sentences if sentence.strip()]

    # 3. 把多个句子组合成 chunk
    chunks=[]
    chunk=[]
    for sentence in sentences:
        sentence += "。"

        # 4. 超长句子单独保存
        if len(sentence) > chunk_size:
            if chunk:
                chunks.append("".join(chunk))
                chunk = []
            chunks.append(sentence)

        # 5. 当前句子可以加入当前块
        elif len("".join(chunk))+len(sentence) <= chunk_size:
            chunk.append(sentence)
            

        # 6. 放不下：保存旧块，当前句子开启新块
        else:
            chunks_last=chunk[-1:]
            chunks.append("".join(chunk))
            chunk=chunks_last + [sentence]
           

    # 7. 保存最后一个 chunk
    if chunk:
        chunks.append("".join(chunk))

    # 8. 返回 chunks
    return chunks     

'''测试代码'''
text = "Python 是一种编程语言。Python 可以用于数据分析。Agent 可以调用工具。RAG 可以检索知识。"

chunks = split_by_sentence(text, 30)

for i, item in enumerate(chunks):
    print(i, len(item), item)


In [ ]:
"""初始化大模型客户端 (用国内的 DeepSeek)"""

import os
import pickle
import math
from openai import OpenAI
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-zh-v1.5",
    local_files_only=True
)

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com"
)


"""构建本地知识库"""
knowledge_base_path = "knowledge_base.pkl"

if os.path.exists(knowledge_base_path):

    # 本地知识库已经存在，直接读取
    with open(knowledge_base_path, "rb") as f:
        knowledge_base = pickle.load(f)

    documents = knowledge_base["documents"]
    embeddings = knowledge_base["embeddings"]

    print("已加载本地知识库，不需要重新向量化。")

else:
    # 本地知识库不存在，才进行向量化

    documents = [
        "Python 是一种编程语言。",
        "Python 可以用于数据分析。",
        "Python 可以用于开发 AI Agent。",
        "Agent 可以通过调用工具完成任务。",
        "RAG 可以检索知识库中的资料。",
    ]

    embeddings = model.encode(documents)

    knowledge_base = {"documents": documents, "embeddings": embeddings}

    with open(knowledge_base_path, "wb") as f:
        pickle.dump(knowledge_base, f)

    print("知识库已构建并保存。")


def cosine_similarity(a, b):
    """相似度计算函数"""

    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)


def search_knowledge_base(
    query, documents, embeddings, model, top_k=3, similarity_threshold=0.6
):
    """根据用户提问检索知识库"""

    # 1,对用户问题进行向量化
    embedding_q = model.encode(query)

    # 2,找到相似度最高的文件（语句）
    results = []

    for i, embedding in enumerate(embeddings):
        score = cosine_similarity(embedding_q, embedding)
        tuple_i = (score, documents[i])
        results.append(tuple_i)

    results.sort(key=lambda x: x[0], reverse=True)

    for score, doc in results:
        print(f"相似度：{score:.4f} | 文档：{doc}")

    # 3，相似度阈值排查
    filtered_results = [item for item in results if item[0] >= similarity_threshold]

    if not filtered_results:
        return "知识库中没有找到足够相关的资料。"

    # 4,选出top_k
    top_results = filtered_results[:top_k]

    # 3，将问题和高匹配度文件交给llm生成回复
    context = "\n".join(x[1] for x in top_results)

    prompt = f"""
    你是一个严谨的知识库问答助手。请遵守以下规则：

    1. 只能根据参考文档中的信息回答问题。
    2. 如果参考文档没有提供足够的信息，请明确回答：
      "根据当前知识库，无法找到足够的信息来回答这个问题。"
    3. 不要使用你自己的已有知识补充答案。
    4. 不要编造参考文档中不存在的事实。

    问题：
    {query}

    参考文档：
    {context}
    """

    response = client.chat.completions.create(
        model="deepseek-chat", messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


print("\n===== 测试 1：有相关资料 =====")
answer = search_knowledge_base(
    query="Python 可以用来做什么？",
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3
)
print("回答：", answer)


# 测试 2：知识库中没有相关资料
print("\n===== 测试 2：没有相关资料 =====")
answer = search_knowledge_base(
    query="太阳系中最大的行星是什么？",
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3
)
print("回答：", answer)

print("\n===== 测试 3：人为提高相似度阈值 =====")

answer = search_knowledge_base(
    query="Python 可以用来做什么？",
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3,
    similarity_threshold=0.99
)

print("最终回答：", answer)


In [ ]:
def retrieve_documents(query,documents,embeddings,model ,top_k=3,similarity_threshold=0.6):
    #1,问题转换成向量
    embedding_q=model.encode(query)

    #2，计算问题与每条文档的余弦相似的
    results=[]

    for i,embedding in enumerate(embeddings):
        score=cosine_similarity(embedding_q,embedding)
        tuple_i=(score,documents[i])
        results.append(tuple_i)

    #3,按相似度从高到低排序
    results.sort(key=lambda x:x[0],reverse=True)

    # 4. 过滤掉低于阈值的文档
    filtered_results=[item for item in  results if item[0]>=similarity_threshold]

    #5,返回前top_k条结果，每条结果包含相似度和文档内容
    return  filtered_results[:top_k]


'''测试代码'''
results = retrieve_documents(
    query="Python 可以用来做什么？",
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3,
    similarity_threshold=0.6
)

if not results:
    print("知识库里没有找到足够相关的资料。")
else:
    print("找到相关资料，可以交给大模型回答。")

    for score, document in results:
        print(f"相似度：{score:.4f} | 文档：{document}")

In [ ]:
def generate_answer(query,results):
    '''生成回答'''

    if not results:
        return f"知识库中没有找到足够相关的资料"

    context="\n".join(x[1] for x in results)

    prompt=f'''
    你是一个严谨的知识库问答助手，请遵守以下规则：
    1. 只能根据参考文档中的信息回答问题。
    2. 如果参考文档没有提供足够的信息，请明确回答："根据当前知识库，无法找到足够的信息来回答这个问题。"
    3. 不要使用已有知识补充参考文档中没有的信息。
    4. 不要编造事实。

    问题 ；
    {query}

    参考文档：
    {context}
    '''

    response=client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

# 测试一：知识库中有相关资料
query = "Python 可以用来做什么？"

results = retrieve_documents(
    query=query,
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3,
    similarity_threshold=0.6
)

answer = generate_answer(query, results)
print("回答：", answer)

# 测试二：知识库中没有相关资料
query = "太阳系中最大的行星是什么？"

results = retrieve_documents(
    query=query,
    documents=documents,
    embeddings=embeddings,
    model=model,
    top_k=3,
    similarity_threshold=0.6
)

answer = generate_answer(query, results)
print("回答：", answer)


In [ ]:
def rag_query(query):

    #1,调用retrieve_documents()检索文档
    results=retrieve_documents(query,documents,embeddings,model ,top_k=3,similarity_threshold=0.6)

    #2，将检索结果交给generate_answer()
    answer = generate_answer(query, results)

    #3,返回最终回答
    return answer

In [ ]:
# 测试 1：知识库中有相关资料
answer = rag_query("Python 可以用来做什么？")
print(answer)

# 测试 2：知识库中没有相关资料
answer = rag_query("太阳系中最大的行星是什么？")
print(answer)

In [ ]:
#测试1：检索结果数量
query="python可以用来做什么"

results=retrieve_documents(query=query,
                       documents=documents,
                       embeddings=embeddings,
                       model=model,
                       top_k=3,
                       similarity_threshold=0.6)

assert len(results) <= 3
print("测试1通过：检索结果数量不超过3条")

#测试2：无相关资料时据答
query="太阳系中最大的行星是什么"

results=retrieve_documents(query=query,
                       documents=documents,
                       embeddings=embeddings,
                       model=model,
                       top_k=3,
                       similarity_threshold=0.6)

assert not results
print("测试2通过：没有相关资料时，检索结果为空")

#3,测试无资料时是否拒答
query="太阳系中最大的行星是什么"

results=[]
answer=generate_answer(query,results)

assert "没有找到足够相关的资料" in answer
print("测试3通过：没有相关资料时据答")

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

from agent import Agent

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

agent = Agent(client)

answer = agent.run("获取一个数字，然后给它乘以5")
print("最终回答：", answer)

In [ ]:
answer = agent.run("再获取一个数字，然后乘以 3")
print("第二轮回答：", answer)